# Chimanimani SBAS pre-flight and MintPy run
Run cells in order. Every check prints PASS / WARN / FAIL and the notebook stops at the first FAIL.

- **Cell 1:** installs, config (AOI from Chimanimani_AOI_v2, limits).
- **Cell 2 (Stage A):** metadata only, no downloads. Confirms the full network is connected, then trims to the best-connected pairs (`MAX_PAIRS`) and confirms again.
- **Cell 3 (Stage B):** downloads one pair at a time, checks the zip is not corrupt, crops to the AOI, checks every crop, deletes the zip. Stops early if the network would split.
- **Cell 4 (Stage C):** MintPy load only, checks the real stack, then runs the rest. MintPy output is shown live.
- **Cell 5:** sanity checks on the final outputs.


In [ ]:
# Google Colab: download the current notebook (.ipynb) to your phone
import json
import datetime
from google.colab import files, _message

# Get the current notebook as JSON
nb_json = _message.blocking_request('get_ipynb', timeout_sec=10)['ipynb']

# Make a filename with date/time
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Colab_Notebook_{stamp}.ipynb"

# Save it inside the Colab VM
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(nb_json, f, ensure_ascii=False, indent=1)

# Trigger download to your device
files.download(filename)

In [ ]:
# ===== CELL 1: install, imports, config =====
!pip -q install hyp3_sdk networkx
import re, os, glob, json, zipfile, shutil, subprocess
from datetime import datetime, timezone
import numpy as np
import networkx as nx
from hyp3_sdk import HyP3
from osgeo import gdal
gdal.UseExceptions()

# ---- CONFIG ----
JOB_NAME       = None                      # HyP3 batch name, or None = all INSAR_GAMMA jobs on the account
EXPECTED_PAIRS = 70
AOI            = (32.7833, -20.0220, 33.1659, -19.6952)   # W, S, E, N (lon/lat, WGS84) from Chimanimani_AOI_v2.shp
RES_M          = 40                        # output pixel size in metres
MIN_VALID      = 0.80                      # a pair needs at least this fraction of valid AOI pixels
ZIP_DIR        = '/content/data/zips'
AOI_DIR        = '/content/data/aoi'
MINTPY_DIR     = '/content/data/mintpy_aoi'
TROPO_METHOD   = 'height_correlation'      # 'height_correlation' or 'pyaps'

def ok(m):   print(f"[PASS] {m}")
def warn(m): print(f"[WARN] {m}")
def fail(m):
    print(f"[FAIL] {m}")
    raise RuntimeError("STOPPED: " + m)

def to_dt(s): return datetime.strptime(s, '%Y%m%d')

def build_graph(edge_list, all_dates):
    G = nx.Graph(); G.add_nodes_from(all_dates); G.add_edges_from(edge_list)
    return G, sorted(nx.connected_components(G), key=len, reverse=True)

def suggest_links(comps):
    order = sorted(comps, key=lambda c: min(c))
    print("Suggested pairs to submit to HyP3 to join the groups:")
    for a, b in zip(order, order[1:]):
        x, y = min(((x, y) for x in a for y in b), key=lambda p: abs((to_dt(p[0]) - to_dt(p[1])).days))
        print(f"   {x} - {y}  (gap {abs((to_dt(x)-to_dt(y)).days)} days)")
W, S, E, N = AOI
print(f"AOI: W {W}, S {S}, E {E}, N {N}  (about {(E-W)*105:.0f} km x {(N-S)*111:.0f} km)")
print("Setup done.")

In [ ]:
# ===== CELL 2: STAGE A - connectivity check, then trim to the best-connected pairs (limit set below) (nothing downloaded) =====
from collections import Counter
import copy

START_DATE         = '20250901'   # ignore the old 2017-2021 test pairs
SUBMIT             = False        # True = actually submit the bridging pairs to HyP3 (uses credits)
LINKS_PER_JUNCTION = 2            # bridging pairs proposed per gap between groups
BRIDGE_NAME        = 'chimanimani-bridge'
MAX_PAIRS          = 40           # maximum number of pairs kept
MIN_LOOPS          = 12           # minimum number of independent loops
MIN_SPAN           = 330          # minimum time span in days

def span_days(ds): return (to_dt(max(ds)) - to_dt(min(ds))).days
def gap_days(e):   return abs((to_dt(e[0]) - to_dt(e[1])).days)

def report(Gx, label):
    """Print PASS/WARN/FAIL for how well connected a network is."""
    n, e = Gx.number_of_nodes(), Gx.number_of_edges()
    print(f"\n--- {label}: {n} dates, {e} pairs ---")
    cs = list(nx.connected_components(Gx))
    if len(cs) == 1: ok("All dates are in ONE connected network")
    else: fail(f"Network is split into {len(cs)} separate groups")
    loops = e - n + 1
    if loops == 0: fail("Network is a tree (no loops), no redundancy")
    elif e / n < 2: warn(f"Low redundancy: {e/n:.1f} pairs per date, {loops} independent loops")
    else: ok(f"Good redundancy: {e/n:.1f} pairs per date, {loops} independent loops")
    br = list(nx.bridges(Gx))
    if br: warn(f"{len(br)} pair(s) are single points of failure: {br}")
    else: ok("No single pair is holding the network together")
    weak = [d for d, k in Gx.degree() if k == 1]
    if weak: warn(f"{len(weak)} date(s) connected by only one pair: {weak}")
    else: ok("Every date has at least two pairs")
    sp = span_days(list(Gx.nodes))
    (ok if sp >= MIN_SPAN else warn)(f"Time span {sp} days ({min(Gx.nodes)} to {max(Gx.nodes)})")

# ---------- 1. read the jobs from HyP3 ----------
if 'hyp3' not in globals(): hyp3 = HyP3(prompt='password')
all_jobs = hyp3.find_jobs(job_type='INSAR_GAMMA', name=JOB_NAME) if JOB_NAME else hyp3.find_jobs(job_type='INSAR_GAMMA')

def scene_date(g): return re.search(r'_(\d{8})T', g).group(1)
def pair_of(j):    return tuple(sorted(scene_date(g) for g in j.job_parameters['granules']))

now = datetime.now(timezone.utc)
def usable(j):
    if not j.succeeded(): return False
    exp = j.expiration_time
    if exp is None: return True
    if exp.tzinfo is None: exp = exp.replace(tzinfo=timezone.utc)
    return exp >= now

jobs = [j for j in all_jobs if usable(j)]
print(f"HyP3 returned {len(all_jobs)} jobs: {len(jobs)} usable, {len(all_jobs)-len(jobs)} failed/running/expired")

# keep only the current stack (drop old experiments)
recent = [j for j in jobs if pair_of(j)[0] >= START_DATE]
print(f"{len(recent)} usable jobs on or after {START_DATE} ({len(jobs)-len(recent)} older ones ignored)")

# keep jobs that include the DEM and the look vectors
n_before = len(recent)
recent = [j for j in recent if j.job_parameters.get('include_dem') and j.job_parameters.get('include_look_vectors')]
print(f"{n_before - len(recent)} jobs dropped because they lack DEM and/or look vectors")
if not recent: fail("No usable jobs include both the DEM and look vectors. MintPy cannot load them.")

# keep jobs that share the most common processing parameters
sig = lambda j: (j.job_parameters.get('looks'), j.job_parameters.get('apply_water_mask'), j.job_parameters.get('include_dem'), j.job_parameters.get('include_look_vectors'))
print("Processing parameter combinations (looks, water_mask, dem, look_vectors):", Counter(sig(j) for j in recent).most_common())
main_sig = Counter(sig(j) for j in recent).most_common(1)[0][0]
recent = [j for j in recent if sig(j) == main_sig]
print(f"Keeping the {len(recent)} jobs that use {main_sig}")

EDGE_JOB, GRANULE = {}, {}
for j in recent:
    EDGE_JOB.setdefault(pair_of(j), j)
    for g in j.job_parameters['granules']: GRANULE[scene_date(g)] = g
ALL_DATES = sorted(GRANULE)
print(f"{len(recent) - len(EDGE_JOB)} duplicate jobs for the same pair ignored")

# ---------- 2. is the full network connected? if not, propose bridging pairs ----------
G, comps = build_graph(list(EDGE_JOB), ALL_DATES)
N, E, C = G.number_of_nodes(), G.number_of_edges(), len(comps)
print(f"\n{N} dates, {E} unique pairs, {C} connected group(s), {E-N+C} independent loops")
for i, c in enumerate(comps, 1):
    print(f"   group {i}: {len(c)} dates, {min(c)} to {max(c)} ({span_days(c)} days)")

if C > 1:
    existing = set(EDGE_JOB)
    new_edges = []
    order = sorted(comps, key=lambda c: min(c))
    print("\nPairs that would join the groups:")
    for a, b in zip(order, order[1:]):
        cands = sorted(((abs((to_dt(x)-to_dt(y)).days), min(x, y), max(x, y)) for x in a for y in b))
        cands = [c for c in cands if (c[1], c[2]) not in existing]
        picked, used_x, used_y = [], set(), set()
        for gap, x, y in cands:
            if x in used_x or y in used_y: continue
            picked.append((gap, x, y)); used_x.add(x); used_y.add(y)
            if len(picked) == LINKS_PER_JUNCTION: break
        for gap, x, y in picked:
            print(f"   {x} - {y}  (gap {gap} days){'  <- long, weak coherence likely' if gap > 60 else ''}")
            new_edges.append((x, y))
    Gp, cp = build_graph(list(EDGE_JOB) + new_edges, ALL_DATES)
    print(f"\nIf these {len(new_edges)} pairs are added: {len(cp)} group(s), {Gp.number_of_nodes()} dates, "
          f"{Gp.number_of_edges()} pairs, span {span_days(ALL_DATES)} days")
    if SUBMIT and len(cp) == 1:
        template = next(iter(EDGE_JOB.values())).to_dict(for_resubmit=True)
        batch = []
        for x, y in new_edges:
            d = copy.deepcopy(template); d['name'] = BRIDGE_NAME
            d['job_parameters']['include_dem'] = True
            d['job_parameters']['include_look_vectors'] = True
            d['job_parameters']['granules'] = [GRANULE[x], GRANULE[y]]
            batch.append(d)
        hyp3.submit_prepared_jobs(batch)
        print(f"Submitted {len(batch)} jobs under the name '{BRIDGE_NAME}'. Wait until they finish, then re-run this cell.")
    else:
        print("Nothing submitted. Set SUBMIT = True and re-run this cell to submit them.")
    fail(f"Network is split into {C} groups. Add the pairs above, wait for HyP3, then re-run this cell.")

# ---------- 3. confirm the FULL network is well connected BEFORE trimming ----------
report(G, "BEFORE TRIMMING (all available pairs)")

# ---------- 4. trim to the best-connected MAX_PAIRS pairs ----------
n_dates0, n_pairs0 = len(ALL_DATES), len(EDGE_JOB)

def pick_dates(T):
    """Drop the least-connected dates one at a time until T remain (None if that is impossible)."""
    keep = set(ALL_DATES)
    while len(keep) > T:
        H = G.subgraph(keep); srt = sorted(keep)
        def gap(d):
            i = srt.index(d)
            return 10**6 if i in (0, len(srt) - 1) else (to_dt(srt[i+1]) - to_dt(srt[i-1])).days
        for d in sorted(keep, key=lambda d: (H.degree(d), gap(d))):
            trial = keep - {d}
            if nx.is_connected(G.subgraph(trial)) and span_days(trial) >= MIN_SPAN:
                keep = trial; break
        else:
            return None
    return keep

def pick_pairs(keep):
    """Shortest-gap spanning chain, then add the shortest extra pairs that remove the most weak links."""
    W = nx.Graph()
    for u, v in G.subgraph(keep).edges(): W.add_edge(u, v, w=gap_days((u, v)))
    S = nx.minimum_spanning_tree(W, weight='w')
    spare = [e for e in W.edges() if not S.has_edge(*e)]
    while S.number_of_edges() < MAX_PAIRS and spare:
        def cost(e):
            T2 = S.copy(); T2.add_edge(*e)
            return (len(list(nx.bridges(T2))), gap_days(e))
        best_e = min(spare, key=cost)
        S.add_edge(*best_e); spare.remove(best_e)
    return S

chosen = None
for T in range(MAX_PAIRS - MIN_LOOPS + 1, 5, -1):     # use as many dates as the limits allow
    keep = pick_dates(T)
    if keep is None: continue
    H = pick_pairs(keep)
    if H.number_of_edges() <= MAX_PAIRS and H.number_of_edges() - len(keep) + 1 >= MIN_LOOPS:
        chosen = (keep, H); break
if chosen is None:
    fail(f"Cannot build a connected network with {MAX_PAIRS} pairs or fewer, {MIN_LOOPS}+ loops and a {MIN_SPAN}+ day span. "
         f"Lower MIN_SPAN or MIN_LOOPS, or raise MAX_PAIRS.")
keep, H = chosen

EDGE_JOB  = {e: EDGE_JOB[e] for e in (tuple(sorted(x)) for x in H.edges())}
ALL_DATES = sorted(keep)
G, comps  = build_graph(list(EDGE_JOB), ALL_DATES)
print(f"\nSELECTED: {G.number_of_nodes()} of {n_dates0} dates and {G.number_of_edges()} of {n_pairs0} pairs "
      f"(limit {MAX_PAIRS}; downloads cut by {n_pairs0 - G.number_of_edges()})")
print(f"   dropped dates: {sorted(set(GRANULE) - keep)}")
print(f"   longest time gap in a pair: {max(gap_days(e) for e in EDGE_JOB)} days")

# ---------- 5. confirm the TRIMMED network is still well connected ----------
report(G, "AFTER TRIMMING (what cell 3 will download)")
if G.number_of_edges() > MAX_PAIRS: fail(f"{G.number_of_edges()} pairs is over the limit of {MAX_PAIRS}")
ok(f"{G.number_of_edges()} pairs, within the limit of {MAX_PAIRS}")

print("\nSTAGE A PASSED: the trimmed network is connected. Stage B confirms every pair is usable over the AOI.")


In [ ]:
# ===== CELL 3: STAGE B - download one pair at a time, check the zip is not corrupt, crop to the AOI, check the crops, delete the zip =====
os.makedirs(ZIP_DIR, exist_ok=True); os.makedirs(AOI_DIR, exist_ok=True)
KINDS    = ['unw_phase', 'corr', 'conncomp', 'dem', 'lv_theta', 'lv_phi', 'water_mask']
REQUIRED = ['unw_phase', 'corr', 'dem', 'lv_theta', 'lv_phi']      # conncomp handled as optional
MIN_FRAC = {'unw_phase': MIN_VALID, 'corr': 0.5, 'conncomp': 0.5, 'dem': 0.9, 'lv_theta': 0.9, 'lv_phi': 0.9}   # min share of real (non-zero) pixels

def clear_zips():
    for z in glob.glob(os.path.join(ZIP_DIR, '*.zip')): os.remove(z)

def get_zip(job):
    """Download one product and verify the zip is intact. Retries once. Returns (path, None) or (None, reason)."""
    why = "unknown"
    for attempt in (1, 2):
        clear_zips()
        try:
            zp = str(job.download_files(ZIP_DIR)[0])
            if os.path.getsize(zp) == 0 or not zipfile.is_zipfile(zp):
                why = "download is empty or not a valid zip"
            else:
                with zipfile.ZipFile(zp) as zf: bad = zf.testzip()      # reads every file and checks its CRC
                if bad is None: return zp, None
                why = f"corrupt file inside zip: {bad}"
        except Exception as ex:
            why = f"download/zip error: {type(ex).__name__}: {ex}"
        print(f"      attempt {attempt} failed: {why}")
    return None, why

def check_tif(path, kind):
    """Open the cropped raster, read every pixel, and make sure it is sane. Returns (share_of_real_pixels, problem_or_None)."""
    ds = gdal.Open(path)
    arr = ds.GetRasterBand(1).ReadAsArray().astype('float64')
    shape, proj = arr.shape, ds.GetProjectionRef(); ds = None
    if arr.size == 0: return 0, "empty raster", shape, proj
    if np.isfinite(arr).mean() < 0.99: return 0, f"{kind}: more than 1% of pixels are NaN/inf (corrupt)", shape, proj
    if kind == 'corr' and np.nanmax(arr) > 1.001: return 0, "corr has values above 1 (corrupt)", shape, proj
    real = float(np.mean(np.isfinite(arr) & (arr != 0)))
    if real < MIN_FRAC[kind]: return real, f"{kind}: only {real:.0%} real pixels over the AOI (need {MIN_FRAC[kind]:.0%})", shape, proj
    return real, None, shape, proj

bad, good, ref_shape, ref_proj = {}, [], None, None
cc_flag, shown_names = {}, False
consecutive_bad = 0

def still_connected(excluded):
    remaining = [e for e in EDGE_JOB if e not in excluded]
    _, cs = build_graph(remaining, ALL_DATES)
    return len(cs) == 1, cs

edges_sorted = sorted(EDGE_JOB)
print(f"Processing {len(edges_sorted)} pairs. Each zip is deleted as soon as its crop has been checked.\n")
for i, e in enumerate(edges_sorted, 1):
    job = EDGE_JOB[e]
    stem = job.files[0]['filename'][:-4]
    out_dir = os.path.join(AOI_DIR, stem); os.makedirs(out_dir, exist_ok=True)
    tag = f"[{i}/{len(edges_sorted)}] {e[0]}-{e[1]}"
    reason, stats = None, {}
    try:
        zpath, reason = get_zip(job)                                   # 1) download + zip integrity check
        if reason is None:
            with zipfile.ZipFile(zpath) as zf:
                names = zf.namelist()
                if not shown_names:
                    print("   files inside one product:", sorted(os.path.basename(n) for n in names)); shown_names = True
                found = {k: next((n for n in names if n.endswith(f'_{k}.tif')), None) for k in KINDS}
                missing = [k for k in REQUIRED if found[k] is None]
                cc_flag[e] = found['conncomp'] is not None
                txt = next((n for n in names if os.path.basename(n) == stem + '.txt'), None)
                if missing: reason = f"missing files in zip: {missing}"
                elif txt is None: reason = "metadata .txt not found in zip"
                else:
                    with zf.open(txt) as s, open(os.path.join(out_dir, stem + '.txt'), 'wb') as d: shutil.copyfileobj(s, d)
        if reason is None:                                             # 2) crop to the AOI
            for k in KINDS:
                if found[k] is None: continue
                gdal.Warp(os.path.join(out_dir, f"{stem}_{k}.tif"), f"/vsizip/{zpath}/{found[k]}",
                          outputBounds=AOI, outputBoundsSRS='EPSG:4326', xRes=RES_M, yRes=RES_M,
                          targetAlignedPixels=True, dstNodata=0, resampleAlg='near' if k == 'conncomp' else 'average')
        if reason is None:                                             # 3) check every cropped file
            for k in REQUIRED + (['conncomp'] if cc_flag.get(e) else []):
                real, problem, shape, proj = check_tif(os.path.join(out_dir, f"{stem}_{k}.tif"), k)
                stats[k] = real
                if problem: reason = problem; break
                if ref_shape is None: ref_shape, ref_proj = shape, proj
                if shape != ref_shape: reason = f"{k}: grid size {shape} differs from {ref_shape}"; break
                if proj != ref_proj: reason = f"{k}: different map projection from the other pairs"; break
    except Exception as ex:
        reason = f"error: {type(ex).__name__}: {ex}"
    finally:
        clear_zips()                                                   # 4) always delete the zip

    if reason:
        bad[e] = reason; consecutive_bad += 1
        shutil.rmtree(out_dir, ignore_errors=True)
        print(f"[FAIL] {tag}: {reason}")
        if consecutive_bad >= 3:
            fail("3 pairs in a row failed. Systematic problem (AOI, login or product type). Fix before continuing.")
        okc, cs = still_connected(set(bad))
        if not okc:
            for k, c in enumerate(cs, 1): print(f"   group {k}: {len(c)} dates, {min(c)} to {max(c)}")
            fail(f"Excluding the failed pair(s) splits the network into {len(cs)} groups. Stopping now to save time.")
        warn("Network is still connected without it, continuing.")
    else:
        consecutive_bad = 0; good.append(e)
        cc_txt = f"conncomp {stats['conncomp']:.0%}" if 'conncomp' in stats else "no conncomp file"
        print(f"[PASS] {tag}: zip intact, crops OK, valid phase {stats['unw_phase']:.0%}, {cc_txt}, grid {ref_shape}")

# ---- final verdict ----
G2, comps2 = build_graph(good, ALL_DATES)
N2, E2, C2 = G2.number_of_nodes(), G2.number_of_edges(), len(comps2)
print(f"\n{len(good)} of {len(edges_sorted)} pairs passed; {len(bad)} rejected")
for e, r in bad.items(): print(f"   rejected {e[0]}-{e[1]}: {r}")
if C2 != 1: fail(f"Final network has {C2} groups.")
ok(f"Final network: {N2} dates, {E2} pairs, ONE connected group, {E2 - N2 + C2} loops, identical grid {ref_shape}")
GOOD_EDGES = good
HAS_CONNCOMP = all(cc_flag.get(e) for e in good)
if HAS_CONNCOMP: ok("All kept pairs have connected-component files: cell 4 will use 'bridging' for unwrapping errors")
else: warn("These products have no connected-component files: cell 4 will use 'phase_closure' for unwrapping errors instead of 'bridging'")
print("\nSTAGE B PASSED: every kept pair is intact, cropped, and the network is connected. Safe to run MintPy.")


In [ ]:
# ===== CELL 4: STAGE C - MintPy load, verify the REAL stack, then run the rest =====
!pip -q install mintpy
import h5py
os.makedirs(MINTPY_DIR, exist_ok=True)
# unwrapping-error method: bridging with conncomp files, phase_closure without
UNW_METHOD = 'bridging' if HAS_CONNCOMP else 'phase_closure'
CC_LINE    = f"{AOI_DIR}/*/*_conncomp.tif" if HAS_CONNCOMP else "no"
print(f"Unwrapping-error correction method: {UNW_METHOD}")
cfg_path = os.path.join(MINTPY_DIR, 'smallbaselineApp.cfg')
open(cfg_path, 'w').write(f"""
mintpy.load.processor        = hyp3
mintpy.load.unwFile          = {AOI_DIR}/*/*_unw_phase.tif
mintpy.load.corFile          = {AOI_DIR}/*/*_corr.tif
mintpy.load.connCompFile     = {CC_LINE}
mintpy.load.demFile          = {AOI_DIR}/*/*_dem.tif
mintpy.load.incAngleFile     = {AOI_DIR}/*/*_lv_theta.tif
mintpy.load.azAngleFile      = {AOI_DIR}/*/*_lv_phi.tif
mintpy.load.waterMaskFile    = {AOI_DIR}/*/*_water_mask.tif
mintpy.reference.minCoherence = 0.7
mintpy.unwrapError.method    = {UNW_METHOD}
mintpy.troposphericDelay.method = {TROPO_METHOD}
mintpy.deramp                = no
mintpy.compute.cluster       = no
""")

def run(args):
    p = subprocess.Popen(['smallbaselineApp.py', cfg_path] + args, cwd=MINTPY_DIR,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end=''); lines.append(line)
    if p.wait() != 0:
        fail(f"smallbaselineApp.py {' '.join(args)} failed. Last lines:\n" + ''.join(lines[-25:]))

def add_connectcomp(stack_path, min_area=2500):
    """Build a connectComponent dataset in the stack from spatially connected blocks of valid unwrapped-phase pixels; blocks under min_area pixels are set to 0."""
    from scipy import ndimage
    with h5py.File(stack_path, 'r+') as f:
        if 'unwrapPhase_phaseClosure' in f: del f['unwrapPhase_phaseClosure']      # remove partial output dataset
        if 'connectComponent' in f: del f['connectComponent']
        n, ny, nx_ = f['unwrapPhase'].shape
        cc = f.create_dataset('connectComponent', shape=(n, ny, nx_), dtype='int16', chunks=True, compression='lzf')
        for i in range(n):
            unw = f['unwrapPhase'][i]
            lab, k = ndimage.label(np.isfinite(unw) & (unw != 0))
            area = np.bincount(lab.ravel(), minlength=k + 1)
            remap = np.zeros(k + 1, dtype='int16'); keep = [j for j in range(1, k + 1) if area[j] >= min_area]
            remap[keep] = np.arange(1, len(keep) + 1)
            cc[i] = remap[lab]
        share = float(np.mean(cc[0] > 0))
    ok(f"Built connectComponent for {n} pairs from the unwrapped phase ({share:.0%} of the first pair is valid)")

# 1) load only (fast)
run(['--end', 'load_data'])

# 2) verify what MintPy actually loaded
with h5py.File(os.path.join(MINTPY_DIR, 'inputs', 'ifgramStack.h5'), 'r') as f:
    pairs = [tuple(sorted(x)) for x in f['date'][:].astype(str)]
    n_drop = int((~f['dropIfgram'][:]).sum())
    has_cc = 'connectComponent' in f
print(f"MintPy loaded {len(pairs)} pairs (expected {len(GOOD_EDGES)})")
if len(pairs) != len(GOOD_EDGES): fail(f"MintPy skipped {len(GOOD_EDGES) - len(pairs)} pair(s). See load_data log above (usually a size mismatch).")
ok("MintPy loaded every pair")
if HAS_CONNCOMP:
    if not has_cc: fail("Pairs have conncomp files but MintPy did not load them, bridging cannot run.")
    ok("Connected-component data present for bridging")
else:
    ok("No conncomp files (expected for these products): using phase_closure, which needs loops in the network")
if n_drop: warn(f"{n_drop} pair(s) flagged as dropped at load time")
Gs, cs = build_graph(pairs, ALL_DATES)
if len(cs) != 1: fail(f"Loaded stack is split into {len(cs)} groups.")
ok(f"Loaded stack is ONE connected network ({Gs.number_of_nodes()} dates, {Gs.number_of_edges()} pairs)")

# 2b) build connectComponent from the unwrapped phase
if not HAS_CONNCOMP:
    add_connectcomp(os.path.join(MINTPY_DIR, 'inputs', 'ifgramStack.h5'))

# 3) remaining MintPy steps
print("\nAll pre-flight checks passed. Running the remaining MintPy steps...")
run(['--start', 'modify_network'])
print("MintPy finished.")

In [ ]:
# ===== CELL 5: sanity checks on the final outputs =====
import h5py
with h5py.File(os.path.join(MINTPY_DIR, 'temporalCoherence.h5'), 'r') as f:
    tc = f['temporalCoherence'][:]
print(f"Temporal coherence: mean {np.nanmean(tc):.2f}, share above 0.7 = {np.nanmean(tc > 0.7):.0%}")
if np.nanstd(tc) < 1e-6: fail("Temporal coherence is flat (blank map). Inversion is not trustworthy.")
ok("Temporal coherence map has real variation")

with h5py.File(os.path.join(MINTPY_DIR, 'velocity.h5'), 'r') as f:
    v = f['velocity'][:] * 1000      # mm/yr
    ry, rx = int(f.attrs['REF_Y']), int(f.attrs['REF_X'])
ny, nx_ = v.shape
# reference pixel check by row and column
if 0 <= ry < ny and 0 <= rx < nx_: ok(f"Reference pixel (row {ry}, col {rx}) is inside the AOI grid ({ny} x {nx_} pixels)")
else: warn(f"Reference pixel (row {ry}, col {rx}) is outside the AOI grid")
print(f"Pixels with a velocity estimate: {np.isfinite(v).mean():.0%}")

p1, p99 = np.nanpercentile(v, [1, 99])
print(f"Velocity 1st to 99th percentile: {p1:.0f} to {p99:.0f} mm/yr")
if max(abs(p1), abs(p99)) > 210: warn("Velocities exceed the range of the comparison studies (about 210 mm/yr). Check for atmosphere or unwrapping errors.")
else: ok("Velocity range is plausible against the literature")
